**04/07/2026**

Just created the 2D scene of an empty warehouse. Robots are currently just rectangles, and they don't have collision. `scene.py` holds only the drawing logic, and `robot.py` holds logic for individual logic only.

First thing to do next is to clean up the clutter in `main.py` and make a `World` class to wrap everything cleanly. Also, we need to separate the physics logic from the rendering. They should run in two independent loops.

**06/07/2026**

Refactored the code and separated everything. `renderer.py` now handles all the drawing logic, `simulation.py` handles the physics, and `collision.py` specifically handles collision. Collision is checked using the **Separating Axis Theorem**. `world.py` updates the scene every tick.

With scalable architecture and collision done, the next step is to have some sort of autonomous driving, like trajectory planning and following.

**09/07/2026**

Refactored the code for future scalability. Added a basic feedback driveToPose. Fixed conflicting variable bug between goal threshold and physics dt making bots stop for no reason. Also implemented naive yield logic so robots can know to not crash into each other.

Anyway, basic autonomous driving achieved. The next step would be to make them avoid walls by driving along them instead of driving headfirst, and improving the yield logic. World and robot needs to be reorganised too.

**10/07/2026**

Ditched the yield logic for potential field-based obstacle avoidance, but still keeping the naive P-controller based to find the goal. Running into some classic local minima and oscillation problems. Perturbation logic kinda working, but it'll need some changes. Gonna draw out the map next maybe and try how one robot moves across that first.

**24/07/2026**

Added shelves. Refactored repulsion code such that it works with any source of obstacles. Cached robot vertices to optimize for compute. Also implemented a spatial grid to further reduce compute for SAT iterations. Overall time complexity went from $O(n^2)$ to $O(n)$.

Clearance code is buggy. Need to find a way next to find the shortest distance between oriented rectangles. Also robots right now can't drive between shelves. We'll need graph-based path-planning. Some sort of graph builder from arbitrary shelf layouts should be implemented.

Tested purely reactive potential fields with varying robot numbers and densities. Will need to code layouts and benchmarks to make performance assessment easier.

Also running into some optimization problem. Will try to optimise the current stack before moving to a better renderer.

Synchronization creates superpositioned failure modes: robots should start sequentially. Maybe a benchmark for this should be implemented.

**26/07/2026**

Implemented the jerk limiter for physical realism. It's important to note that the jerk limiter must be added before the repulsion layer. If not, the robot will not react fast enough and crash.

Implemented tangential force component in the potential field-based navigation. Robots now navigate head-on obstacles much faster.

When graphification comes, the nodes should stay away from obstacles. The robots shouldn't feel any potential whatsoever during transportation between nodes (or minimal potential forces). Those are just for worst case scenario, reacting against obstacles. When taking inventory from shelves or when docking, robots should move more slowly, more carefully, and the potential field will be retuned to more conservative gains.

Robot-to-robot collision doesn't work very well with potential fields. Might want to look into ORCA (Optimal Reciprocal Collision Avoidance)

Between graph nodes we would want to have a trajectory planner. There, we could profile the motion.

Torn between implementing docking and shelving first, or graphification first.

Also, at the time of writing, for a 20m x 40m ware house with 9 shelves, $n=38$ robots is the upper bound before they start crashing/oscillating/producing deadlocks.

**27/07/2026**

COMPLETELY DELETED ALL PREVIOUS CODE.

Gonna build this again from the ground up in pygame with a completely different architecture. Matplotlib real time playback fails at $n \geq 10$ robots, and using ffmpeg fails at $n \geq 30$ robots at timeframes of more than 60 seconds. We now move to PyGame renderer, and we're going to optimize math again. Will start doing that tomorrow.

**01/08/2026**

Starting to move up the abstraction layer. Building a graph of the warehouse is hard. In the world space, we're considering building disconnected subgraphs between objects, then using a connector function to connect everything into a connected graph. After that, graph traversal algorithms will be used.

Then the planning layer becomes graph building and traversing between nodes. The lower layer becomes potential based navigation for static objects (docks and shelves), and ORCA for robot-robot navigation. 

**06/08/2026**

Using AABB (Axis Aligned Bounding Box) checks for shelf potential fields because the general case with oriented rectangles require too much trignonometry. With 80 shelves, simulation time is now approximately 1x real time. Will implement spatial hashing next. Theoretically this could bring us back to 20x real time with good implementation.

Also, after verification, collision checks might be disabled to optimise for runtime.

**08/08/2026**

Fully vectorised shelves, docks, and pallets. Did try spatial hashing in the form of grids. Asymptotically it behaves like $\mathcal{O}(nk)$ yes, but it's still very bad. Potential fields just are too expensive for $n$-body simulations: they require even OBB-to-OBB distance to be smooth and accurate.

Even with spatial hashing, we only moved from 1x real time to 5x real time. This is not enough for long term work: we would want to maintain 10-30x real time speed at minimum.

Turning off potential field obstacle avoidance for now. Working on construction of the graph. Idea is sound, but implementation requires rigorous calculations.

Using DFS at first for graph traversal. Also using an FSM-flavoured controller for this. Will need to look into the concept of lookahead soon: currently we're seeing unwanted behaviour: if we go from nodes $A \rightarrow B \rightarrow C$ where $A$, $B$, and $C$ are collinear, the controller will go from $A$ then stop at $B$, then go from $B$ and stop at $C$. We need to maintain velocity to go straight from $A$ to $C$.

**12/08/2026**
After lots of experiment runs, we have a few findings:

